In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
%load_ext autoreload
%autoreload 2

## Dataset

In [3]:
import time
from torch_geometric.nn.pool.decimation import decimation_indices


def decimate(tensors, ptr, decimation_factor: int):
    """Decimates each element of the given tuple of tensors."""
    idx_decim, ptr_decim = decimation_indices(ptr, decimation_factor)
    tensors_decim = tuple(tensor[idx_decim] for tensor in tensors)
    return tensors_decim, ptr_decim


def benchmark(func, *args, iterations=100):
    start_time = time.time()
    for _ in range(iterations):
        _ = func(*args)
    end_time = time.time()
    return (end_time - start_time) / iterations

In [4]:
B, N, D = 128, 1000, 3  # Example dimensions
tensors = (torch.randn(B * N, D),)
ptr = torch.tensor([0] + [N] * B, dtype=torch.long).cumsum(0)
decimation_factor = 2

benchmark(decimate, tensors, ptr, decimation_factor)

0.0017751455307006836

In [5]:
import torch
import time
from typing import Tuple, Sequence

def decimation_indices(lengths: torch.LongTensor, decimation_factor: float) -> Tuple[torch.Tensor, torch.LongTensor]:
    if decimation_factor < 1:
        raise ValueError(
            f"The argument `decimation_factor` should be higher than (or "
            f"equal to) 1 for downsampling. (got {decimation_factor})")

    batch_size = lengths.size(0)
    decim_lengths = torch.div(lengths, decimation_factor, rounding_mode='floor')
    decim_lengths.clamp_(min=1)  # Prevent empty examples

    decim_indices = [
        lengths[i] + torch.randperm(lengths[i], device=lengths.device)[:decim_lengths[i]]
        for i in range(batch_size)
    ]
    decim_indices = torch.cat(decim_indices, dim=0)
    return decim_indices, decim_lengths


def decimation_indices(lengths: torch.LongTensor, decimation_factor: float) -> torch.Tensor:
    if decimation_factor < 1:
        raise ValueError(
            f"The argument `decimation_factor` should be higher than (or "
            f"equal to) 1 for downsampling. (got {decimation_factor})")

    batch_size = lengths.size(0)
    lengths.clamp_(min=1)
    decim_lengths = torch.div(lengths, decimation_factor, rounding_mode='floor').clamp(min=1)
    max_decim_length = decim_lengths.max().item()
    decim_indices = torch.full((batch_size, max_decim_length), -1, dtype=torch.long, device=lengths.device)
    for i in range(batch_size):
        sampled_indices = torch.randperm(lengths[i], device=lengths.device)[:decim_lengths[i]]
        decim_indices[i, :sampled_indices.size(0)] = sampled_indices

    return decim_indices, decim_lengths


def decimate(tensors: Tuple[torch.Tensor, ...], lengths: torch.LongTensor, decimation_factor: float) -> Tuple[torch.Tensor, ...]:
    idx_decim, ptr_decim = decimation_indices(lengths, decimation_factor)
    tensors_decim = tuple(tensor[idx_decim] for tensor in tensors)
    return tensors_decim, ptr_decim

In [6]:
B, N, D = 128, 1000, 3  # Example dimensions
tensors = (torch.randn(B, N, D),)
lengths = torch.tensor([N] * B, dtype=torch.long)
lengths[:50] = 4
decimation_factor = 3

decimation_indices(lengths, decimation_factor)
# decimate(tensors, lengths, decimation_factor)

(tensor([[  3,  -1,  -1,  ...,  -1,  -1,  -1],
         [  0,  -1,  -1,  ...,  -1,  -1,  -1],
         [  2,  -1,  -1,  ...,  -1,  -1,  -1],
         ...,
         [396, 237, 789,  ..., 426, 471, 358],
         [737, 357, 511,  ..., 540, 863, 653],
         [669,  80, 963,  ..., 789, 524, 252]]),
 tensor([  1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1, 333, 333, 333, 333, 333, 333,
         333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333,
         333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333,
         333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333,
         333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333,
         333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 3

In [7]:
B, N, D = 128, 1000, 3  # Example dimensions
tensors = (torch.randn(B, N, D),)
lengths = torch.tensor([10] * B)
decimation_factor = 2

# decimation_indices(lengths, decimation_factor)
# benchmark(decimation_indices, lengths, decimation_factor)
benchmark(decimate, tensors, lengths, decimation_factor)

0.0011762261390686035

### RandLANet

In [8]:
from torch_pointcloud.models.randlanet import RandLANet, FPModule, shared_mlp2d


model = RandLANet(num_features=3, num_classes=20)

In [9]:
xyz = torch.rand(2, 1024, 3).cuda()
features = torch.rand(2, 3, 1024).cuda()
model.cuda()

out = model(xyz, features)
print(f"{out.shape = }")

out.shape = torch.Size([2, 20, 1024])


In [24]:
from torch_pointcloud.models.randlanet import RandLANetClassification
from torch_geometric.nn import MLP


model = RandLANetClassification(num_features=3, num_classes=20)

In [25]:
xyz = torch.rand(2, 1024, 3).cuda()
features = torch.rand(2, 3, 1024).cuda()
model.cuda()

out = model(xyz, features)
print(f"{out.shape = }")

b1_xyz.shape=torch.Size([2, 256, 3])
b1_feat.shape=torch.Size([2, 32, 256])
b1_lengths.shape=torch.Size([2])
b2_xyz.shape=torch.Size([2, 64, 3])
b2_feat.shape=torch.Size([2, 128, 64])
b2_lengths.shape=torch.Size([2])
x.shape=torch.Size([2, 128])
out.shape = torch.Size([2, 20])
